# Day 3 · 변경 라인에 근거를 남기는 코드 리뷰 Agent

화면을 따라 실행하되, 결과를 자동 게시하지 않습니다. 모든 외부 쓰기는 dry-run과 사람 승인을 먼저 거칩니다.

In [1]:
# 최초 1회 설치. 이미 설치했다면 빠르게 완료됩니다.
%pip install -q -r ../../requirements-day1.txt
# STT 실습을 실제 음성으로 실행할 때만 다음 줄의 주석을 해제합니다.
# %pip install -q -r ../../requirements-stt-optional.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print({"workspace": str(ROOT), "python": sys.version.split()[0]})

{'workspace': '/Users/sungjae-cha/sungjae-cha/llm-agent-and-workflow-automation', 'python': '3.12.12'}


## 1. unified diff를 읽고 추가 라인 번호를 복원합니다

In [3]:
from src.course_services.review_service import parse_unified_diff, run_review_service

diff_path = ROOT / "data/day3_review_cases/unsafe_pr.diff"
diff_text = diff_path.read_text(encoding="utf-8")
parsed = parse_unified_diff(diff_text)
print(parsed.changed_paths)
print([(line.line, line.text) for line in parsed.added_lines])

('src/payment_job.py',)
[(9, '    result = eval(command)'), (10, '    try:'), (11, '        requests.post("https://example.invalid/jobs", json=payload)'), (12, '    except Exception:'), (13, '        return {"status": "ok"}'), (14, '    return result')]


## 2. 결정론적 baseline으로 고위험 finding을 만듭니다

In [4]:
review = run_review_service(diff_text)
print(json.dumps(review, ensure_ascii=False, indent=2))
assert all(finding["line"] in {line.line for line in parsed.added_lines} for finding in review["findings"])
assert review["automatic_publish"] is False

{
  "status": "SUCCESS",
  "findings": [
    {
      "path": "src/payment_job.py",
      "line": 9,
      "severity": "P0",
      "title": "검증되지 않은 문자열이 코드로 실행될 수 있음",
      "body": "입력 경로에 따라 임의 코드 실행으로 이어질 수 있습니다.",
      "evidence": "result = eval(command)",
      "suggestion": "허용된 명령이나 파서만 호출하는 명시적 registry로 교체하세요.",
      "confidence": 0.99,
      "rule_id": "unsafe-dynamic-execution"
    },
    {
      "path": "src/payment_job.py",
      "line": 11,
      "severity": "P1",
      "title": "외부 쓰기 전에 사람 승인 경계가 보이지 않음",
      "body": "재시도나 잘못된 대상 선택이 실제 게시·수정으로 이어질 수 있습니다.",
      "evidence": "requests.post(\"https://example.invalid/jobs\", json=payload)",
      "suggestion": "dry-run payload, 대상 확인, human_approved 조건을 쓰기 호출 앞에 두세요.",
      "confidence": 0.91,
      "rule_id": "external-write-without-approval"
    },
    {
      "path": "src/payment_job.py",
      "line": 12,
      "severity": "P2",
      "title": "넓은 예외 처리가 실패 계약을 지움",
      "body": "인증 오류와 입력 오류가 같은 경로로 숨겨져 복구 결정을

## 3. Codex에게 맡길 작업도 scope와 test 계약부터 씁니다

In [5]:
from src.course_services.codex_harness import CodexTaskSpec, render_codex_task

spec = CodexTaskSpec(
    objective="새 review rule 하나와 정상·실패 test를 추가한다.",
    allowed_paths=("src/course_services", "tests"),
    acceptance_tests=("python -m pytest -q tests/test_course_services.py",),
)
print(render_codex_task(spec))

목표
새 review rule 하나와 정상·실패 test를 추가한다.

먼저 읽을 파일
- AGENTS.md
- README.md

변경 허용 범위
- src/course_services
- tests

완료 조건
- python -m pytest -q tests/test_course_services.py

금지 행동
- secret 출력 또는 commit
- workspace 밖 파일 변경
- 사람 승인 없는 외부 쓰기
- 통과를 위한 기존 test 약화

복구 지점
- 작업 전 commit

마지막에 변경 파일, 핵심 diff, 실행한 test 명령과 결과를 보고하세요.


## 완료 확인

- Day 3 결과 JSON을 확인했습니다.
- 실패 경로가 traceback 대신 `error_code`로 남는지 확인했습니다.
- 외부 쓰기와 자동 메일이 발생하지 않았음을 확인했습니다.
- 변경한 코드는 diff와 test 결과를 사람이 검토합니다.